# Ensemble methods. Exercises


In this section we have two exercises:

1. Find the best three classifier in the stacking method using the classifiers from scikit-learn package.

2. Build arcing arc-x4 method.

In [41]:
# imports
import pickle
import numpy as np

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

from sklearn.metrics import accuracy_score

from itertools import combinations

from sklearn.preprocessing import StandardScaler

In [42]:
# download data
from google.colab import drive
drive.mount('/content/drive')

with open("/content/drive/MyDrive/Colab-Notebooks/Machine-Learning/Lab-6/data_set.pkl", "rb") as f:
    unscaled_data_set = pickle.load(f)
with open("/content/drive/MyDrive/Colab-Notebooks/Machine-Learning/Lab-6/labels.pkl", "rb") as f:
    labels = pickle.load(f)
with open("/content/drive/MyDrive/Colab-Notebooks/Machine-Learning/Lab-6/test_data_set.pkl", "rb") as f:
    unscaled_test_data_set = pickle.load(f)
with open("/content/drive/MyDrive/Colab-Notebooks/Machine-Learning/Lab-6/test_labels.pkl", "rb") as f:
    test_labels = pickle.load(f)
with open("/content/drive/MyDrive/Colab-Notebooks/Machine-Learning/Lab-6/unique_labels.pkl", "rb") as f:
    unique_labels = pickle.load(f)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [43]:
# preprocessing
scaler = StandardScaler()
data_set = scaler.fit_transform(unscaled_data_set)
test_data_set = scaler.transform(unscaled_test_data_set)

## Exercise 1: Find the best three classifier in the stacking method

Please use the following classifiers:

* Linear regression,
* Nearest Neighbors,
* Linear SVM,
* Decision Tree,
* Naive Bayes,
* QDA.

In [44]:
available_models = {
    "Linear Regression": LinearRegression(),
    "Nearest Neighbors": KNeighborsClassifier(),
    "Linear SVM": SVC(kernel='linear'),
    "Decision Tree": DecisionTreeClassifier(),
    "Naive Bayes": GaussianNB(),
    "QDA": QuadraticDiscriminantAnalysis()
}

In [45]:
# stacked classifier
def build_stacked_classifier(classifiers):
    output = []
    for classifier in classifiers:
        preds = classifier.predict(data_set)
        if isinstance(classifier, LinearRegression):
            preds = np.round(np.clip(preds, 0, 1))
        output.append(preds)

    stacked_data_train = np.array(output).T

    stacked_classifier = LogisticRegression()
    stacked_classifier.fit(stacked_data_train, labels.reshape(-1,))

    test_set = []
    for classifier in classifiers:
        test_preds = classifier.predict(test_data_set)
        if isinstance(classifier, LinearRegression):
            test_preds = np.round(np.clip(test_preds, 0, 1))
        test_set.append(test_preds)

    test_set = np.array(test_set).reshape((len(test_set[0]),3))
    predicted = stacked_classifier.predict(test_set)

    return predicted

In [46]:
# check which three models have best accuracy, iterate on combinations of 3
threes = list(combinations(available_models.keys(), 3))

results = []

for t in threes:
    current_models = [available_models[name] for name in t]

    for clf in current_models:
        clf.fit(data_set, labels.ravel())

    predicted = build_stacked_classifier(current_models)
    acc = accuracy_score(test_labels, predicted)

    results.append((acc, t))

results.sort(key=lambda x: x[0], reverse=True)

print("Top 5 best threes:")
for score, names in results[:5]:
    print(f"Accuracy: {score:.4f} for models: {names}")

Top 5 best threes:
Accuracy: 0.9500 for models: ('Linear SVM', 'Decision Tree', 'Naive Bayes')
Accuracy: 0.9500 for models: ('Linear SVM', 'Decision Tree', 'QDA')
Accuracy: 0.9000 for models: ('Nearest Neighbors', 'Linear SVM', 'Naive Bayes')
Accuracy: 0.9000 for models: ('Nearest Neighbors', 'Linear SVM', 'QDA')
Accuracy: 0.9000 for models: ('Nearest Neighbors', 'Decision Tree', 'Naive Bayes')


In [47]:
def build_chosen_classifiers():
    # Linear SVM, Decision Tree & QDA should have best accuracy
    svm = SVC(kernel='linear').fit(data_set, labels)
    dt = DecisionTreeClassifier().fit(data_set, labels)
    qda = QuadraticDiscriminantAnalysis().fit(data_set, labels)

    return [svm, dt, qda]

In [48]:
# check chosen three models
chosen_classifiers = build_chosen_classifiers()
predicted = build_stacked_classifier(chosen_classifiers)
accuracy = accuracy_score(test_labels, predicted)
print(accuracy)

0.95


## Exercise 2:

Use the boosting method and change the code to fullfill the following requirements:

* the weights should be calculated as:
$w_{n}^{(t+1)}=\frac{1+ I(y_{n}\neq h_{t}(x_{n})}{\sum_{i=1}^{N}1+I(y_{n}\neq h_{t}(x_{n})}$,
* the prediction is done with a voting method.

In [49]:
# prepare data set
def generate_data(sample_number, feature_number, label_number):
    data_set = np.random.random_sample((sample_number, feature_number))
    labels = np.random.choice(label_number, sample_number)
    return data_set, labels

labels = 8
dimension = 3
test_set_size = 1000
train_set_size = 5000
train_set, train_labels = generate_data(train_set_size, dimension, labels)
test_set, test_labels = generate_data(test_set_size, dimension, labels)

# init weights
number_of_iterations = 10
weights = np.ones((test_set_size,)) / test_set_size


def train_model(classifier, weights):
    return classifier.fit(X=test_set, y=test_labels, sample_weight=weights)

def calculate_error(model):
    predicted = model.predict(test_set)
    I=calculate_accuracy_vector(predicted, test_labels)
    Z=np.sum(I)
    return (1+Z)/1.0

Fill the two functions below:

In [50]:
def set_new_weights(model):
    # I(y_n != h_t(x_n))
    incorrect = (model.predict(test_set) != test_labels).astype(int)
    return (1 + incorrect) / np.sum(1 + incorrect)

Train the classifier with the code below:

In [51]:
classifier = DecisionTreeClassifier(max_depth=1, random_state=1)
classifier.fit(X=train_set, y=train_labels)
alphas = []
classifiers = []
for iteration in range(number_of_iterations):
    model = train_model(classifier, weights)
    weights = set_new_weights(model)
    classifiers.append(model)

# print(weights)

Set the validation data set:

In [52]:
validate_x, validate_label = generate_data(1, dimension, labels)

Fill the prediction code:

In [53]:
# collecting votes
def get_prediction(x):
    votes = np.array([m.predict(x) for m in classifiers])
    predictions = []
    for i in range(x.shape[0]):
        predictions.append(np.argmax(np.bincount(votes[:, i])))
    return np.array(predictions)

Test it:

In [54]:
prediction = get_prediction(validate_x)[0]

print(prediction)

3
